# 🚀 **AIE4ML Tutorial: From QKeras → hls4ml → AMD AI Engine**

This tutorial shows how to:

* Build a small **quantized QKeras** model
* Convert to **hls4ml (bit-exact)**
* Convert to **AIE4ML (bit-exact)**
* Compare **x86 simulation output**
* Apply **simple AIE tuning overrides** (parallelism, tiling, placement)
* Inspect **AIE simulation reports**

---

## Environment Setup

```bash
conda create -n aie4ml_env python=3.10 -y && conda activate aie4ml_env
pip install "tensorflow==2.15.*" "tensorflow-model-optimization==0.7.5" qkeras hls4ml
pip install aie4ml pyparsing ipykernel
```

# 1️⃣ Setup & Imports

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.initializers import RandomNormal

from qkeras import QDense, QActivation, quantized_bits, quantized_relu

import hls4ml

np.random.seed(42)
tf.random.set_seed(42)

2026-03-16 12:59:32.791195: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-16 12:59:32.819004: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-16 12:59:32.819023: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-16 12:59:32.820019: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-16 12:59:32.824811: I tensorflow/core/platform/cpu_feature_guar

In [2]:
import os
import sys
import subprocess
vitis_settings = "~/tools/Xilinx/2025.2/Vitis/settings64.sh"
#vitis_settings = "/opt/Xilinx/Vitis/2024.1/settings64.sh"

proc = subprocess.run(
    ["bash", "-lc", f"source {vitis_settings} >/dev/null 2>&1 && env -0"],
    capture_output=True,
)

if proc.returncode != 0:
    raise RuntimeError(
        f"Failed to source Vitis settings: {vitis_settings}\n"
        + proc.stderr.decode("utf-8", errors="ignore")
    )
# Apply environment exported by: source /opt/Xilinx/Vivado/2024.1/settings64.sh
# (proc is already present in this notebook and contains `env -0` output)
for entry in proc.stdout.split(b"\x00"):
    if not entry or b"=" not in entry:
        continue
    key, value = entry.split(b"=", 1)
    os.environ[key.decode("utf-8", errors="ignore")] = value.decode("utf-8", errors="ignore")

# Keep the active conda env's tools and shared libraries ahead of system paths.
# This is needed for helpers like readelf during the Vitis build flow.
env_prefix = sys.prefix
env_bin = os.path.join(env_prefix, "bin")
env_lib = os.path.join(env_prefix, "lib")
os.environ["PATH"] = f"{env_bin}:{os.environ.get('PATH', '')}"
os.environ["LD_LIBRARY_PATH"] = f"{env_lib}:{os.environ.get('LD_LIBRARY_PATH', '')}".rstrip(":")

# Verify v++ is now visible to this Python process
check = subprocess.run(
    ["bash", "-lc", "command -v v++ && v++ --version | head -n 1"],
    capture_output=True,
    text=True
)
print(check.stdout if check.returncode == 0 else check.stderr)

/home/z.ma/tools/Xilinx/2025.2/Vitis/bin/v++




# 2️⃣ Build a Small QKeras Model

Currently supports MLP-style architectures, like Dense → ReLU → Dense → ….

In [3]:
IN_FEATURES = 128
HIDDEN = 256
OUT_FEATURES = 64

def build_qkeras_model(in_features=128, hidden=256, out_features=64):
    model = Sequential([
        QActivation(quantized_bits(8, 2), name="input_quant", input_shape=(in_features,)),
        QDense(hidden,
               name="qfc1",
               kernel_quantizer=quantized_bits(8,0,alpha=1),
               bias_quantizer=quantized_bits(8,2,alpha=1),
               bias_initializer=RandomNormal(mean=0.0, stddev=0.05)),
        QActivation(quantized_relu(8,0), name="qrelu1"),
        QDense(out_features,
               name="qfc2",
               kernel_quantizer=quantized_bits(8,0,alpha=1),
               bias_quantizer=quantized_bits(8,2,alpha=1),
               bias_initializer=RandomNormal(mean=0.0, stddev=0.05)),
        QActivation(quantized_bits(8,2), name="output_quant"),
    ])
    return model

model = build_qkeras_model()

model.compile(optimizer=Adam(1e-3), loss="mse")

model.summary()

2026-03-16 12:59:35.021319: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-16 12:59:35.024588: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-16 12:59:35.027713: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_quant (QActivation)   (None, 128)               0         
                                                                 
 qfc1 (QDense)               (None, 256)               33024     
                                                                 
 qrelu1 (QActivation)        (None, 256)               0         
                                                                 
 qfc2 (QDense)               (None, 64)                16448     
                                                                 
 output_quant (QActivation)  (None, 64)                0         
                                                                 
Total params: 49472 (193.25 KB)
Trainable params: 49472 (193.25 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


# 3️⃣  Convert: Baseline hls4ml + AIE models

We create two compiled projects:

 🔹 `proj_hls/` – reference bit-exact model
 
 🔹 `proj_aie/` – AIE-backend model



In [5]:
# Create HLS config from model
cfg = hls4ml.utils.config_from_keras_model(model, granularity='name')

hls_model = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=cfg,
    output_dir='proj_hls',
    project_name='proj_hls',
    bit_exact=True,
)

# You can specify the batch size and number of graph iterations for AIE backend
BATCH = 8
ITERS = 10
PLATFORM = 'xilinx_vek280_base_202520_1'

aie_model = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=cfg,
    output_dir='proj_aie',
    backend='aie',
    project_name='proj_aie',
    batch_size=BATCH,
    iterations=ITERS,
    part = PLATFORM
)

hls_model.compile()
aie_model.compile()

print("Models compiled.")

v++ --compile --config aie.cfg --mode aie --target=x86sim  --include=src/kernels --include=src/kernels/dense_bias_relu --include=src/weights --include=src --platform=/home/z.ma/tools/Xilinx/2025.2/Vitis/base_platforms/xilinx_vek280_base_202520_1/xilinx_vek280_base_202520_1.xpfm app.cpp --aie.workdir=./Work_x86 --aie.output-archive=libadf_x86.a |& tee log

****** v++ v2025.2 (64-bit)
  **** SW Build 6295257 on 2025-11-13-01:29:13
  **** Start of session at: Mon Mar 16 12:59:39 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

INFO: [v++ 60-1548] Creating build summary session with primary output /home/z.ma/aie-pl-final/aie4ml/tutorials/proj_aie/Work_x86/Work_x86.aiecompiler_summary, at Mon Mar 16 12:59:42 2026
INFO: [v++ 82-10534] Setting option --aie.num-trace-streams to value 4
INFO: [v++ 82-31] Launching aiecompiler: aiecompiler --config /home/z.ma/aie-pl-final/aie4ml/tutorials/proj_aie/Work

# 4️⃣ Bit-Exact Check (HLS vs AIE x86)

We test the first output batch.

The AIE simulator may emit more samples if the graph has multiple iterations.


In [6]:
def compare_bit_exact(hls4ml_model, aie4ml_model, sim_mode = 'x86'):
    x = np.random.random((BATCH, IN_FEATURES)).astype(np.float32)
    y_hls = hls4ml_model.predict(x)
    y_aie = aie4ml_model.predict(x, simulator=sim_mode)[:BATCH]

    mse = np.mean((y_hls - y_aie)**2)
    mae = np.mean(np.abs(y_hls - y_aie))
    max_diff = np.max(np.abs(y_hls - y_aie))

    print("MSE       :", mse)
    print("MAE       :", mae)
    print("Max |diff|:", max_diff)

# compare bit-exactness on the AIE x86 simulator output
compare_bit_exact(hls_model, aie_model)

x86simulator --pkg-dir=./Work_x86 |& tee -a log

  DEPRECATION NOTICE:
  External Traffic Generator flows will be deprecated in future releases
  Library: XTLM IPC

INFO: Reading options file './Work_x86/options/x86sim.options'.
Simulation completed successfully returning zero
MSE       : 0.0004520416259765625
MAE       : 0.01446533203125
Max |diff|: 0.03125


# 5️⃣ Build the model

Compile the aie_model in `aie` mode to generate the AIE hardware design.

In [7]:
print("Building AIE project...")

aie_model.build()

print("AIE build & compile completed.")

# compare bit-exactness on the AIE HW simulator output
compare_bit_exact(hls_model, aie_model, sim_mode = 'aie')


Building AIE project...
v++ --compile --config aie.cfg --mode aie --target=hw  --include=src/kernels --include=src/kernels/dense_bias_relu --include=src/weights --include=src --platform=/home/z.ma/tools/Xilinx/2025.2/Vitis/base_platforms/xilinx_vek280_base_202520_1/xilinx_vek280_base_202520_1.xpfm app.cpp --aie.workdir=./Work --aie.output-archive=libadf.a |& tee log

****** v++ v2025.2 (64-bit)
  **** SW Build 6295257 on 2025-11-13-01:29:13
  **** Start of session at: Mon Mar 16 13:00:14 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

INFO: [v++ 60-1548] Creating build summary session with primary output /home/z.ma/aie-pl-final/aie4ml/tutorials/proj_aie/Work/Work.aiecompiler_summary, at Mon Mar 16 13:00:16 2026
INFO: [v++ 82-10534] Setting option --aie.num-trace-streams to value 4
INFO: [v++ 82-10534] Setting option --aie.enable-partition to value 0:38
INFO: [v++ 82-31] Launching aiecompiler

# 6️⃣ View AIE Simulation Report

The report includes:

* reports on output interval and throughput (across all out ports)
* ports, memory, AIE core usage, and others


In [8]:
from aie4ml.simulation import read_aie_report

report = read_aie_report(aie_model)
report

{'throughput': {'Avg_GOPs': 1410.613,
  'Min_GOPs': 1412.414,
  'Max_GOPs': 1404.343},
 'output_interval': {'global': {'min_ns': 556.8,
   'max_ns': 560.0,
   'avg_ns': 557.511,
   'samples': 9},
  'per_port': {'y_p0.txt': {'min_ns': 556.8,
    'max_ns': 560.0,
    'avg_ns': 557.511,
    'samples': 9}}},
 'AIE_info': ['Mapping Analysis Report',
  '',
  '',
  'Acronym List',
  '    * CR(x,y)    - Core <column, row>',
  '    * MG(x,y):b  - MemoryGroup <column, row> : Bank',
  '    * MT(x,y):b  - MemoryTile <column, row> : Bank',
  '',
  '',
  '===============================================',
  'Block Mapping Report:',
  '===============================================',
  '',
  'Block:Function Name  CR(x,y)/IO(x)  Schedule  Utilization  Variable Name  Graph Name',
  '-------------------  -------------  --------  -----------  -------------  ----------',
  'i0:run               CR(7,0)               0        1.000  kk[0]          dut       ',
  'i1:run               CR(7,1)               

# 7️⃣ Apply Tuning Overrides (Parallelism, Tiling, Placement)

AIE4ML lets users **override hardware choices** per layer.

### Example knobs:
* Number of parallel cascade chains (`cas_num`)
* Length of each cascade (`cas_length`)
* Tiling sizes (`tile_m`, `tile_n`, `tile_k`)
* AIE tile placement (`row`, `col`)


In [9]:
def tune_first_dense(cfg):
    layers = list(cfg['LayerName'].keys())
    dense_like = [l for l in layers if 'dense' in l.lower() or 'fc' in l.lower()]
    target = dense_like[0]

    cfg['LayerName'][target].update({

        # ⚙️ Parallelism:
        #   - cas_num splits the *output features* (N dimension)
        #   - cas_length splits the *input features*  (K dimension)
        # Higher parallelism = more AIE tiles used = higher throughput. Try to keep both <= 8.
        'parallelism': {'cas_num': 2, 'cas_length': 2},

        # 🧩 Tiling: how the GEMM is partitioned inside the AIE. Default is usually optimal
        # Controls tile sizes along M (batch), K (input features), N (output features).
        'tiling': {'tile_m': 4, 'tile_k': 8, 'tile_n': 8},

        # 📍 Placement: hard-pin the AIE layer graph to start from a specific tile (row, col).
        'placement': {'row': 0, 'col': 10},
    })

    print("Tuned:", target)
    return target

tuned_layer = tune_first_dense(cfg)


Tuned: qfc1


# 8️⃣ Convert Tuned AIE Model

We keep the same model but pass **the updated hls_config**.

➡️ *Tip:* You can now test impact on bit-exactness and performance.

In [10]:
aie_model_tuned = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=cfg,
    output_dir='proj_aie_tuned',
    backend='aie',
    project_name='proj_aie_tuned',
    batch_size=BATCH,
    iterations=ITERS,
    part=PLATFORM
)

aie_model_tuned.write()
aie_model_tuned.build()

compare_bit_exact(hls_model, aie_model_tuned, sim_mode = 'aie')

read_aie_report(aie_model_tuned)

v++ --compile --config aie.cfg --mode aie --target=hw  --include=src/kernels --include=src/kernels/dense_bias_relu --include=src/weights --include=src --platform=/home/z.ma/tools/Xilinx/2025.2/Vitis/base_platforms/xilinx_vek280_base_202520_1/xilinx_vek280_base_202520_1.xpfm app.cpp --aie.workdir=./Work --aie.output-archive=libadf.a |& tee log


/home/z.ma/miniconda3/envs/aie4ml/lib/python3.10/site-packages/keras/src/constraints.py:365: UserWarning: The `keras.constraints.serialize()` API should only be used for objects of type `keras.constraints.Constraint`. Found an instance of type <class 'qkeras.quantizers.quantized_bits'>, which may lead to improper serialization.
  warnings.warn(



****** v++ v2025.2 (64-bit)
  **** SW Build 6295257 on 2025-11-13-01:29:13
  **** Start of session at: Mon Mar 16 13:01:55 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

INFO: [v++ 60-1548] Creating build summary session with primary output /home/z.ma/aie-pl-final/aie4ml/tutorials/proj_aie_tuned/Work/Work.aiecompiler_summary, at Mon Mar 16 13:01:58 2026
INFO: [v++ 82-10534] Setting option --aie.num-trace-streams to value 4
INFO: [v++ 82-10534] Setting option --aie.enable-partition to value 0:38
INFO: [v++ 82-31] Launching aiecompiler: aiecompiler --config /home/z.ma/aie-pl-final/aie4ml/tutorials/proj_aie_tuned/Work/aie_hw.cfg
                                           AI Engine Compiler
                                     Version 2025.2 (linux64-bit)
                                     SW Build 6299465 on 2025-11-14-04:57:17
                          Copyright 1986-2022 Xilinx, Inc. All 

{'throughput': {'Avg_GOPs': 1613.303,
  'Min_GOPs': 1616.842,
  'Max_GOPs': 1606.275},
 'output_interval': {'global': {'min_ns': 486.4,
   'max_ns': 489.6,
   'avg_ns': 487.467,
   'samples': 9},
  'per_port': {'y_p0.txt': {'min_ns': 486.4,
    'max_ns': 489.6,
    'avg_ns': 487.467,
    'samples': 9}}},
 'AIE_info': ['Mapping Analysis Report',
  '',
  '',
  'Acronym List',
  '    * CR(x,y)    - Core <column, row>',
  '    * MG(x,y):b  - MemoryGroup <column, row> : Bank',
  '    * MT(x,y):b  - MemoryTile <column, row> : Bank',
  '',
  '',
  '===============================================',
  'Block Mapping Report:',
  '===============================================',
  '',
  'Block:Function Name  CR(x,y)/IO(x)  Schedule  Utilization  Variable Name  Graph Name',
  '-------------------  -------------  --------  -----------  -------------  ----------',
  'i0:run               CR(10,0)              0        1.000  kk[0]          dut       ',
  'i1:run               CR(11,0)              

# 🎉 Tutorial Complete!

